# EuroSAT embedding selection ablation

This notebook selects an encoder representation for frozen linear probing on the official GEO-Bench `m-eurosat` splits. Candidate selection uses **training and validation data only**. The official test split is loaded only after a winner has been selected.

The notebook answers two separate questions:

1. Should tokens be taken before or after the encoder's final LayerNorm?
2. Should the fixed-size embedding use CLS, mean spatial pooling, all-token pooling, CLS + spatial mean, or CLS + metadata-prefix tokens?

## Context & Methods

### Candidates

| Pooling | Definition | Dimension | Interpretation |
|---|---|---:|---|
| `cls` | Final CLS token | 144 | Global representation learned by the encoder |
| `mean_fine` | Mean of all fine spatial patch tokens | 144 | Resolution-independent scene representation |
| `mean_all` | Mean of metadata, CLS, and fine tokens | 144 | Secondary ablation; includes missing-metadata embeddings on EuroSAT |
| `cls_mean_fine` | Concatenation of CLS and mean-fine | 288 | Tests complementary global and spatial information with modest extra probe capacity |
| `cls_metadata` | CLS followed by all flattened metadata tokens | 720 for this checkpoint | Tests whether the fixed prefix tokens accumulate useful image context |

Each pooling is evaluated using both `pre_norm` and `post_norm` tokens. All candidates come from the same deterministic encoder forward pass for each batch. The encoder remains frozen.

GEO-Bench EuroSAT does not provide `lat`, `lon`, `month`, or `era5` to this model. Its four metadata positions therefore begin from the learned missing-metadata embeddings. `cls_metadata` tests whether those prefix tokens acquire scene information through encoder attention; it does **not** measure the value of real geographic metadata.

### Key assumptions

- The official GEO-Bench train, validation, and test partitions are available locally.
- The checkpoint and its resolved pretraining configuration belong to the same run.
- Candidate ranking uses mean validation average accuracy over identical seeds.
- Candidates within `tie_tolerance` of the best mean score are treated as tied. The documented priority favors raw pre-norm CLS for literature comparability, then pre-norm mean-fine for a general-purpose embedding.
- Test metrics are descriptive final metrics, not part of representation selection.

### Why flattened spatial tokens are excluded

Flattening spatial tokens gives every patch position separate classifier weights, greatly increases probe capacity, requires fixed spatial dimensions, and is not comparable to a 144-dimensional CLS or mean embedding. Applying one shared linear head to each token and averaging logits is mathematically equivalent to averaging the tokens before that same linear head. `cls_mean_fine` is the controlled spatial alternative. `cls_metadata` flattens only the fixed-count prefix tokens, so its dimension remains independent of image resolution, but its larger 720-dimensional probe must still be considered when interpreting gains.

## 1. Parameters

The defaults use the complete EuroSAT2k splits and the published-style 50-epoch linear-probe setting. Set limits for a quick local smoke test. Cache files include checkpoint file metadata and sample limits to reduce accidental reuse.

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

default_run_dir = repo_root
run_dir = Path(os.environ.get('MEOX_RUN_DIR', default_run_dir))
config_path = Path(os.environ.get(
    'MEOX_CONFIG', repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
))
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
geobench_root = Path(os.environ['GEO_BENCH_DIR'])
cache_dir = run_dir / 'analysis/geobench_eurosat_embedding_selection'

dataset_name = 'm-eurosat'
partition_name = 'default'
train_limit = None
valid_limit = None
test_limit = None
batch_size = 128
num_workers = 4
sample_seed = 42
device_override = None
reuse_cache = True

probe_epochs = 50
probe_batch_size = 1024
probe_learning_rate = 5e-2
probe_seeds = (0, 1, 2, 3, 4)
tie_tolerance = 0.005  # 0.5 percentage points
winner_override = None  # Example: 'pre_norm__mean_fine'
evaluate_selected_on_test = True

## 2. Load the frozen encoder and selection splits

In [ ]:
import json
import random
import sys
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

sys.path.insert(0, str(repo_root))

from datasets.geobench import GeoBenchClassificationDataset
from utils.extract_embeddings import (
    build_model_from_config,
    load_config,
    make_inference_dataloader,
    maybe_limit_dataset,
    model_input_schema,
    move_to_device,
    resolve_device,
)
from utils.linear_probe import classification_metrics

assert config_path.exists(), f'Missing config: {config_path}'
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'
assert (geobench_root / 'classification_v1.0' / dataset_name).exists(), (
    f'Missing GEO-Bench EuroSAT data under {geobench_root}'
)

config = load_config(str(config_path))
device = resolve_device(device_override)
_, model_band_names, _ = model_input_schema(config)

def build_split(split_name, limit):
    base = GeoBenchClassificationDataset(
        root_dir=str(geobench_root), dataset_name=dataset_name, split=split_name,
        model_band_names=model_band_names, partition_name=partition_name,
    )
    return base, maybe_limit_dataset(base, limit, sample_seed)

train_base, train_dataset = build_split('train', train_limit)
valid_base, valid_dataset = build_split('valid', valid_limit)
model = build_model_from_config(config, str(checkpoint_path), device)

print(f'device={device} train={len(train_dataset)} valid={len(valid_dataset)}')
print('input bands:', valid_base.raster_band_names)
print('source mapping:', valid_base.band_mapping)
print('preprocessing:', valid_base.preprocessing_signature)
print('checkpoint:', checkpoint_path)

## 3. Extract every candidate in one pass

The model is in evaluation mode and routing is deterministic. For each token source, the three native pooling strategies use the model's own pooling implementation. `cls_mean_fine` concatenates the corresponding CLS and mean-fine vectors. `cls_metadata` concatenates CLS with the flattened metadata-prefix tokens in the model's configured metadata order.

In [ ]:
candidate_order = [
    'pre_norm__cls',
    'post_norm__cls',
    'pre_norm__mean_fine',
    'post_norm__mean_fine',
    'pre_norm__mean_all',
    'post_norm__mean_all',
    'pre_norm__cls_mean_fine',
    'post_norm__cls_mean_fine',
    'pre_norm__cls_metadata',
    'post_norm__cls_metadata',
]
candidate_labels = {
    key: key.replace('__', ' + ').replace('_', ' ') for key in candidate_order
}
selection_priority = [
    'pre_norm__cls',
    'pre_norm__mean_fine',
    'post_norm__cls',
    'post_norm__mean_fine',
    'pre_norm__mean_all',
    'post_norm__mean_all',
    'pre_norm__cls_mean_fine',
    'post_norm__cls_mean_fine',
    'pre_norm__cls_metadata',
    'post_norm__cls_metadata',
]

def embeddings_from_features(active_model, features):
    candidates = {}
    for token_source in ('pre_norm', 'post_norm'):
        prefix_tokens = (
            features['pre_norm_meta_cls_tokens']
            if token_source == 'pre_norm'
            else features['meta_cls_tokens']
        )
        metadata_tokens = prefix_tokens[:, :-1]
        cls = active_model.encoder._pool_feature_tokens(
            features, token_source=token_source, pooling='cls'
        )
        mean_fine = active_model.encoder._pool_feature_tokens(
            features, token_source=token_source, pooling='mean_fine'
        )
        mean_all = active_model.encoder._pool_feature_tokens(
            features, token_source=token_source, pooling='mean_all'
        )
        candidates[f'{token_source}__cls'] = cls
        candidates[f'{token_source}__mean_fine'] = mean_fine
        candidates[f'{token_source}__mean_all'] = mean_all
        candidates[f'{token_source}__cls_mean_fine'] = torch.cat(
            [cls, mean_fine], dim=-1
        )
        candidates[f'{token_source}__cls_metadata'] = torch.cat(
            [cls, metadata_tokens.flatten(start_dim=1)], dim=-1
        )
    return candidates

@torch.inference_mode()
def extract_candidates(active_model, dataset, dataset_info, selected_candidates):
    dataloader = make_inference_dataloader(dataset, batch_size, num_workers)
    collected = {name: [] for name in selected_candidates}
    labels = []
    sample_ids = []
    active_model.eval()
    use_amp = device.type == 'cuda'

    for batch in tqdm(dataloader, desc='Extracting candidate embeddings'):
        rasters = move_to_device(batch['raster_dict'], device)
        validity = move_to_device(batch.get('raster_valid_masks'), device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            features = active_model.forward_features(
                raster_dict=rasters,
                raster_valid_masks=validity,
                raster_band_names=dataset_info.raster_band_names,
                return_routing=False,
                stochastic_routing=False,
            )
            batch_candidates = embeddings_from_features(active_model, features)
            for name in selected_candidates:
                collected[name].append(batch_candidates[name].float().cpu())
        labels.append(batch['label'].cpu())
        sample_ids.extend(str(value) for value in batch['sample_id'])

    outputs = {name: torch.cat(values).numpy() for name, values in collected.items()}
    outputs['labels'] = torch.cat(labels).numpy()
    outputs['sample_ids'] = np.asarray(sample_ids, dtype=str)
    return outputs

def load_or_extract(split_name, dataset, dataset_info, selected_candidates, limit):
    cache_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_stat = checkpoint_path.stat()
    checkpoint_tag = (
        f'{checkpoint_path.stem}_{checkpoint_stat.st_size}_{checkpoint_stat.st_mtime_ns}'
    )
    limit_tag = 'full' if limit is None else str(limit)
    candidate_tag = 'all' if list(selected_candidates) == candidate_order else selected_candidates[0]
    cache_path = cache_dir / (
        f'{dataset_name}_{split_name}_{dataset_info.preprocessing_signature}_'
        f'{checkpoint_tag}_{limit_tag}_{candidate_tag}.npz'
    )
    if reuse_cache and cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as archive:
            outputs = {name: archive[name] for name in archive.files}
        if all(name in outputs for name in selected_candidates):
            print('loaded:', cache_path)
            return outputs
    outputs = extract_candidates(model, dataset, dataset_info, selected_candidates)
    np.savez_compressed(cache_path, **outputs)
    print('saved:', cache_path)
    return outputs

In [ ]:
train_outputs = load_or_extract(
    'train', train_dataset, train_base, candidate_order, train_limit
)
valid_outputs = load_or_extract(
    'valid', valid_dataset, valid_base, candidate_order, valid_limit
)

shape_rows = []
for candidate in candidate_order:
    train_values = train_outputs[candidate]
    valid_values = valid_outputs[candidate]
    assert train_values.ndim == 2 and valid_values.ndim == 2
    assert train_values.shape[1] == valid_values.shape[1]
    assert np.isfinite(train_values).all() and np.isfinite(valid_values).all()
    shape_rows.append({
        'candidate': candidate_labels[candidate],
        'dimension': train_values.shape[1],
        'train_samples': train_values.shape[0],
        'valid_samples': valid_values.shape[0],
    })
display(pd.DataFrame(shape_rows).set_index('candidate'))

## 4. Validation-only linear-probe ablation

For each candidate and seed, one linear layer is trained on the official training split. The epoch with the highest validation average accuracy is retained. All candidates use exactly the same optimizer, learning rate, batch size, epochs, and seeds. No test samples are loaded in this section.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.inference_mode()
def predict_logits(head, features):
    batches = DataLoader(
        torch.from_numpy(features).float(), batch_size=probe_batch_size
    )
    return torch.cat([head(batch.to(device)).cpu() for batch in batches]).numpy()

def train_candidate_probe(candidate, train_data, valid_data, test_data=None):
    train_features = train_data[candidate]
    valid_features = valid_data[candidate]
    train_labels = train_data['labels'].astype(np.int64).reshape(-1)
    valid_labels = valid_data['labels'].astype(np.int64).reshape(-1)
    num_classes = int(max(train_labels.max(), valid_labels.max()) + 1)
    train_dataset_for_probe = TensorDataset(
        torch.from_numpy(train_features).float(),
        torch.from_numpy(train_labels).long(),
    )
    rows = []

    for seed in probe_seeds:
        seed_everything(int(seed))
        head = nn.Linear(train_features.shape[1], num_classes).to(device)
        optimizer = torch.optim.AdamW(
            head.parameters(), lr=probe_learning_rate, weight_decay=0.0
        )
        loss_fn = nn.CrossEntropyLoss()
        generator = torch.Generator().manual_seed(int(seed))
        train_loader = DataLoader(
            train_dataset_for_probe, batch_size=probe_batch_size,
            shuffle=True, generator=generator,
        )
        best_score = float('-inf')
        best_epoch = 0
        best_metrics = None
        best_state = None

        for epoch in range(1, probe_epochs + 1):
            head.train()
            for features, labels in train_loader:
                logits = head(features.to(device))
                loss = loss_fn(logits, labels.to(device))
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            head.eval()
            valid_logits = predict_logits(head, valid_features)
            metrics = classification_metrics(valid_logits, valid_labels, multilabel=False)
            if metrics['average_accuracy'] > best_score:
                best_score = metrics['average_accuracy']
                best_epoch = epoch
                best_metrics = metrics
                best_state = deepcopy(head.state_dict())

        row = {
            'candidate_key': candidate,
            'candidate': candidate_labels[candidate],
            'seed': int(seed),
            'dimension': int(train_features.shape[1]),
            'best_epoch': int(best_epoch),
            **{f'valid_{name}': value for name, value in best_metrics.items()},
        }
        if test_data is not None:
            head.load_state_dict(best_state)
            head.eval()
            test_labels = test_data['labels'].astype(np.int64).reshape(-1)
            test_logits = predict_logits(head, test_data[candidate])
            test_metrics = classification_metrics(test_logits, test_labels, multilabel=False)
            row.update({f'test_{name}': value for name, value in test_metrics.items()})
        rows.append(row)
    return rows

In [ ]:
validation_rows = []
for candidate in tqdm(candidate_order, desc='Candidate probes'):
    validation_rows.extend(
        train_candidate_probe(candidate, train_outputs, valid_outputs)
    )
validation_runs = pd.DataFrame(validation_rows)

validation_summary = (
    validation_runs.groupby(['candidate_key', 'candidate', 'dimension'], sort=False)
    .agg(
        mean_valid_aa=('valid_average_accuracy', 'mean'),
        std_valid_aa=('valid_average_accuracy', 'std'),
        mean_valid_oa=('valid_overall_accuracy', 'mean'),
        mean_valid_macro_f1=('valid_macro_f1', 'mean'),
        mean_best_epoch=('best_epoch', 'mean'),
    )
    .reset_index()
)
validation_summary['ci95_valid_aa'] = (
    1.96 * validation_summary['std_valid_aa'] / np.sqrt(len(probe_seeds))
)
validation_summary = validation_summary.sort_values(
    'mean_valid_aa', ascending=False
).reset_index(drop=True)
validation_summary.index = validation_summary.index + 1
validation_summary.index.name = 'rank'
display(validation_summary.round(4))

cache_dir.mkdir(parents=True, exist_ok=True)
validation_runs.to_csv(cache_dir / 'validation_runs.csv', index=False)
validation_summary.to_csv(cache_dir / 'validation_summary.csv', index=True)

In [ ]:
best_mean = validation_summary['mean_valid_aa'].max()
tied_keys = [
    key for key in candidate_order
    if key in set(
    validation_summary.loc[
        validation_summary['mean_valid_aa'] >= best_mean - tie_tolerance,
        'candidate_key',
    ]
    )
]
if winner_override is not None:
    if winner_override not in candidate_order:
        raise ValueError(f'Unknown winner_override: {winner_override}')
    selected_candidate = winner_override
else:
    selected_candidate = next(key for key in selection_priority if key in tied_keys)

print(f'Best validation mean AA: {best_mean:.4f}')
print(f'Tied within {tie_tolerance:.3f}: {[candidate_labels[key] for key in tied_keys]}')
print('Selected candidate:', candidate_labels[selected_candidate])

plot_table = validation_summary.sort_values('mean_valid_aa')
fig, axis = plt.subplots(figsize=(9, 5.5))
colors = [
    '#C84B31' if key == selected_candidate else '#176B87'
    for key in plot_table['candidate_key']
]
axis.barh(
    plot_table['candidate'], plot_table['mean_valid_aa'],
    xerr=plot_table['std_valid_aa'].fillna(0), color=colors, alpha=0.9, capsize=3,
)
axis.axvline(best_mean - tie_tolerance, color='#555555', linestyle='--', linewidth=1)
axis.set_xlabel('Validation average accuracy')
axis.set_title('Frozen linear-probe representation ablation')
axis.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

seed_comparison = validation_runs.pivot(
    index='seed', columns='candidate', values='valid_average_accuracy'
)
display(seed_comparison.round(4))

## 5. Final test evaluation

This section constructs the official test dataset only after `selected_candidate` is fixed. Only that representation is extracted and evaluated. Rerunning the notebook with a manually chosen `winner_override` is valid only when the override was decided without inspecting test results.

In [ ]:
if evaluate_selected_on_test:
    test_base, test_dataset = build_split('test', test_limit)
    test_outputs = load_or_extract(
        'test', test_dataset, test_base, [selected_candidate], test_limit
    )
    final_runs = pd.DataFrame(
        train_candidate_probe(
            selected_candidate, train_outputs, valid_outputs, test_outputs
        )
    )
    test_metric_columns = [
        'test_average_accuracy', 'test_overall_accuracy', 'test_macro_f1'
    ]
    final_summary = pd.DataFrame({
        'metric': test_metric_columns,
        'mean': [final_runs[column].mean() for column in test_metric_columns],
        'std': [final_runs[column].std(ddof=1) for column in test_metric_columns],
    })
    display(final_runs.round(4))
    display(final_summary.set_index('metric').round(4))
    final_runs.to_csv(cache_dir / 'selected_candidate_test_runs.csv', index=False)

    result_manifest = {
        'checkpoint': str(checkpoint_path.resolve()),
        'preprocessing_signature': train_base.preprocessing_signature,
        'selected_candidate': selected_candidate,
        'selection_metric': 'validation_average_accuracy',
        'tie_tolerance': tie_tolerance,
        'probe_epochs': probe_epochs,
        'probe_learning_rate': probe_learning_rate,
        'probe_seeds': list(probe_seeds),
        'split_sizes': {
            'train': len(train_dataset), 'valid': len(valid_dataset), 'test': len(test_dataset)
        },
        'test_aggregate': final_summary.set_index('metric').to_dict(orient='index'),
    }
    with open(cache_dir / 'selected_candidate_results.json', 'w', encoding='utf-8') as handle:
        json.dump(result_manifest, handle, indent=2)
else:
    print('Test evaluation disabled. Selection is complete:', candidate_labels[selected_candidate])

## 6. How to interpret the result

1. First inspect `validation_summary`. The highest mean validation average accuracy is the empirical winner.
2. Use the per-seed table to verify that a result is not driven by one lucky seed.
3. Treat candidates inside the dashed 0.5-point tie region as practically tied unless more seeds give a stable separation.
4. If CLS wins, the encoder's learned global token is effective for scene classification. If mean-fine wins, class information is distributed more reliably across spatial tokens.
5. If pre-norm wins, the final LayerNorm removes useful magnitude or direction information. If post-norm wins consistently, normalization improves transfer for this task.
6. If `cls_metadata` wins, remember that EuroSAT provides no actual metadata: the result would show that missing-metadata prefix tokens accumulated image context. A small gain does not justify its 720-dimensional linear head.
7. If `cls_mean_fine` wins only slightly, weigh that gain against its doubled feature dimension and larger linear head. Report it as a pooling ablation rather than the default literature-comparable result.
8. Do not choose a representation from the test table. The test table estimates the performance of the already selected representation.

### Frozen repository decision

The ablation grid is closed. The repository-wide embedding protocol is **`pre_norm` + `mean_fine`**, producing a 144-dimensional embedding for the S checkpoint. The four spatial-mean candidates were within the predefined 0.5-point validation tie tolerance, so `pre_norm_mean_fine` was selected by the documented simplicity priority rather than by test performance. Its five-seed EuroSAT test average accuracy was 89.70% (standard deviation 0.33 percentage points).

Future downstream datasets evaluate how well this fixed representation generalizes; they do not reopen token-source or pooling selection. CLS, CLS + metadata, CLS + mean-fine, mean-all, post-norm, and embedding postprocessing remain ablations only.